# IntZ Example 18: IntZ → Z1 High-z Bridge ⭐ EPS Science
**EPS Research — Flynn & Cannaliato (2025)**

Connect IntZ (z~0.9, Hα) to Z1 (z~5, [CII]) to show the full negative-omega regime spanning z~0.6 to z~5.68.

This notebook requires the Z1 corpus to also be loaded.

In [ ]:
# ── Colab setup: auto-download corpus from Zenodo ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import urllib.request
    CORPORA = {
        'intz_corpus_v1b.json': 'https://zenodo.org/records/20453189/files/intz_corpus_v1b.json',
        'intz_corpus_v1b_flat.csv': 'https://zenodo.org/records/20453189/files/intz_corpus_v1b_flat.csv',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")
    print("Ready.")
else:
    print("Running locally — corpus files loaded from working directory.")


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

# Load IntZ corpus
with open('/home/david/Documents/RAG Project/Z=2 RAG/Zenodo/intz_corpus_v1b.json') as f:
    data = json.load(f)
galaxies = data['galaxies']
print(f"Loaded {len(galaxies)} galaxies")

In [ ]:
import statistics, os

# IntZ omega
omega_intz = [g['omega']['omega_value_rad_gyr'] for g in galaxies
              if g['omega']['omega_available']
              and g['omega']['omega_value_rad_gyr'] is not None]
z_intz = [g['redshift']['z_spec'] for g in galaxies
          if g['omega']['omega_available']
          and g['omega']['omega_value_rad_gyr'] is not None]

# Z1 omega (load if available)
Z1_PATH = os.path.expanduser(
    '~/Documents/rag-corpus-series/examples/highz/high_z_kinematic_corpus_Z1.json')
omega_z1, z_z1 = [], []
if os.path.exists(Z1_PATH):
    with open(Z1_PATH) as f:
        z1data = json.load(f)
    z1gals = z1data if isinstance(z1data, list) else z1data.get('galaxies', [])
    for g in z1gals:
        om = g.get('omega', {})
        if om.get('omega_available') and om.get('omega_value_rad_gyr') is not None:
            omega_z1.append(om['omega_value_rad_gyr'])
            z_z1.append(g.get('redshift', {}).get('z_spec', 5.0))
    print(f"Z1 loaded: {len(z1gals)} galaxies, {len(omega_z1)} with omega")
else:
    print("Z1 corpus not found at expected path — using reference values")
    omega_z1 = [-13.05]
    z_z1 = [5.0]

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(z_intz, omega_intz, c='firebrick', alpha=0.3, s=10,
           label=f'IntZ KROSS T1 (N={len(omega_intz)})')
if len(omega_z1) > 1:
    ax.scatter(z_z1, omega_z1, c='purple', s=60, zorder=5,
               label=f'Z1 ALPINE T1 (N={len(omega_z1)})')
else:
    ax.scatter([5.0], [-13.05], c='purple', s=100, zorder=5,
               marker='*', label='Z1 median reference')
ax.axhline(0, color='gray', ls='-', lw=1, alpha=0.5)
ax.axhline(7.06, color='green', ls=':', lw=2, label='z=0 SPARC +7.06')
ax.set_xlabel('Redshift z')
ax.set_ylabel('ω (rad/Gyr)')
ax.set_title('EPS Research: Negative Omega Regime z~0.6 to z~5.7\n'
             'IntZ + Z1 Combined')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('intz_nb18_intz_z1_bridge.png', dpi=120)
plt.show()
print(f"IntZ median ω = {statistics.median(omega_intz):.3f} rad/Gyr")
print(f"Z1 reference  = -13.05 rad/Gyr")